# Giai đoạn 1 - CLIP + FAISS trên 1.200 sản phẩm điện tử ABO
Notebook này thay dataset thời trang bằng tập con điện tử từ Amazon Berkeley Objects. Không train CLIP; chỉ dùng model pretrained để sinh embedding.

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas matplotlib pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, shutil, time
import faiss, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sentence_transformers import SentenceTransformer

print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available(): print('Tên GPU:', torch.cuda.get_device_name(0))

In [ ]:
zip_path = Path('/content/drive/MyDrive/KhoaLuan/abo_electronics_colab_1200.zip')
work_dir = Path('/content/abo_electronics')
if work_dir.exists(): shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)
shutil.unpack_archive(zip_path, work_dir)
dataset_dir = work_dir / 'abo_electronics_colab'
df = pd.read_csv(dataset_dir / 'products_metadata.csv')
df['image_path'] = df['image_path'].map(lambda p: str(dataset_dir / p))
assert len(df) == 1200
assert df['productId'].is_unique and df['externalId'].is_unique
assert df['image_path'].map(lambda p: Path(p).is_file()).all()
print(df['category'].value_counts())
df.head()

In [ ]:
preview = df.groupby('category', group_keys=False).sample(n=1, random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax in axes.flat: ax.axis('off')
for ax, (_, row) in zip(axes.flat, preview.iterrows()):
    ax.imshow(Image.open(row['image_path']).convert('RGB'))
    ax.set_title(f"{row['category']}\n{row['name'][:45]}", fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
image_model = SentenceTransformer('sentence-transformers/clip-ViT-B-32', device=device)
text_model = SentenceTransformer('sentence-transformers/clip-ViT-B-32-multilingual-v1', device=device)
print('Thiết bị:', device, '- số chiều:', image_model.get_sentence_embedding_dimension())

In [ ]:
images = [Image.open(path).convert('RGB') for path in df['image_path']]
start = time.perf_counter()
embeddings = image_model.encode(images, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
encode_seconds = time.perf_counter() - start
assert embeddings.shape == (1200, 512)
print('Embedding:', embeddings.shape, '- thời gian:', round(encode_seconds, 2), 'giây')

In [ ]:
product_ids = df['productId'].astype('int64').to_numpy()
base_index = faiss.IndexFlatIP(embeddings.shape[1])
index = faiss.IndexIDMap2(base_index)
index.add_with_ids(embeddings, product_ids)
assert index.ntotal == 1200
print('FAISS:', type(index).__name__, '- vectors:', index.ntotal)

In [ ]:
query_row = df[df['category'] == 'audio'].iloc[0]
query_image = Image.open(query_row['image_path']).convert('RGB')
start = time.perf_counter()
query_vector = image_model.encode([query_image], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
image_scores, image_ids = index.search(query_vector, 10)
image_ms = (time.perf_counter() - start) * 1000
image_results = pd.DataFrame({'productId': image_ids[0], 'score': image_scores[0]})
image_results = image_results.merge(df[['productId','name','category','image_path']], on='productId')
print('Ảnh truy vấn:', query_row['name'])
print('responseTimeMs:', round(image_ms, 2))
image_results[['productId','score','name','category']]

In [ ]:
query_text = 'tai nghe không dây màu đen'
start = time.perf_counter()
text_vector = text_model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
text_scores, text_ids = index.search(text_vector, 10)
text_ms = (time.perf_counter() - start) * 1000
text_results = pd.DataFrame({'productId': text_ids[0], 'score': text_scores[0]})
text_results = text_results.merge(df[['productId','name','category','image_path']], on='productId')
print('Truy vấn:', query_text, '- responseTimeMs:', round(text_ms, 2))
text_results[['productId','score','name','category']]

In [ ]:
output_dir = Path('/content/drive/MyDrive/KhoaLuan/abo_electronics_phase1')
output_dir.mkdir(parents=True, exist_ok=True)
np.save(output_dir / 'abo_electronics_1200_embeddings.npy', embeddings)
faiss.write_index(index, str(output_dir / 'abo_electronics_1200_idmap.faiss'))
df[['productId','externalId','name','category','productType','brand','color']].to_csv(output_dir / 'abo_electronics_1200_mapping.csv', index=False)
evidence = {
    'dataset': 'ABO Electronics Subset', 'count': len(df), 'dimension': embeddings.shape[1],
    'faissIndex': 'IndexIDMap2(IndexFlatIP)', 'imageTopK': 10,
    'imageResponseTimeMs': round(image_ms, 2), 'textQuery': query_text,
    'textResponseTimeMs': round(text_ms, 2), 'encodeSeconds': round(encode_seconds, 2)
}
(output_dir / 'phase1_evidence.json').write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', output_dir)
print(json.dumps(evidence, ensure_ascii=False, indent=2))